In [ ]:
from browser_use import Agent, ChatBrowserUse
from dotenv import load_dotenv
import asyncio

load_dotenv()

async def main():
    llm = ChatBrowserUse()
    task = """
    Найди мне черный чай на Ozon.ru и выведи мне первые пять позиций в формате:
    1. Название товара
    2. Цена
    3. Ссылка на товар
    """
    agent = Agent(task=task, llm=llm)
    result = await agent.run()
    print(result)
await main()

INFO     [Agent] 🔗 Found URL in task: https://Ozon.ru, adding as initial action...
INFO     [Agent] 🎯 Task: 
    Найди мне черный чай на Ozon.ru и выведи мне первые пять позиций в формате:
    1. Название товара
    2. Цена
    3. Ссылка на товар
    
INFO     [Agent] Starting a browser-use agent with version 0.11.2, with provider=browser-use and model=bu-1-0
INFO     [Agent]   ▶️   navigate: url: https://Ozon.ru, new_tab: False
INFO     [tools] 🔗 Navigated to https://Ozon.ru
INFO     [Agent] 

INFO     [Agent] 📍 Step 1:
INFO     [Agent]   🧠 Memory: The page is still loading, as indicated by the blank screen and the few generic interactive elements. I need to wait for the page to fully load to find the search bar and proceed with searching for "черный чай" (black tea). I will wait for a few seconds.
INFO     [Agent]   ▶️   wait: seconds: 3
INFO     [tools] 🕒 waited for 3 seconds
INFO     [Agent] 

INFO     [Agent] 📍 Step 2:
INFO     [Agent]   🧠 Memory: The page has loaded successfully.

Я пробовал добавить Agno. У меня не получилось так как OpenRouter просил кредиты закинуть на аккаунт, пробовал через Siliconflow проблема почему-то с ключом.

In [ ]:
import asyncio
import os
from openai import AsyncOpenAI
from rich.console import Console
from rich.panel import Panel
from rich.text import Text
from rich.markdown import Markdown
from dotenv import load_dotenv

load_dotenv()

API_KEY = "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"
BASE_URL = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-4o-mini"

console = Console()

class Agent:
    def __init__(self, client: AsyncOpenAI, name: str, system_prompt: str, color: str):
        self.client = client
        self.name = name
        self.system_prompt = system_prompt
        self.color = color

    async def reply(self, history: list) -> str:
        messages = [{"role": "system", "content": self.system_prompt}] + history
        try:
            response = await self.client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
                temperature=0.7,
                max_tokens=500
            )
            content = response.choices[0].message.content
            return content or "..."
        except Exception as e:
            return f"Ошибка: {e}"


async def run_discussion(topic: str, rounds: int = 2):
    client = AsyncOpenAI(api_key=API_KEY, base_url=BASE_URL)

    agents = [
        Agent(client, "Оптимист", 
              "Ты вечный оптимист. Находишь хорошее во всём. Отвечай кратко (2-3 предложения) на русском.", 
              "cyan"),
        Agent(client, "Пессимист", 
              "Ты пессимист. Видишь потенциальные недостатки и риски. Отвечай кратко (2-3 предложения) на русском.", 
              "magenta"),
        Agent(client, "Учёный", 
              "Ты учёный. Анализируешь на основе фактов и логики. Отвечай кратко (2-3 предложения) на русском.", 
              "yellow"),
        Agent(client, "Комик", 
              "Ты комик. Шутишь обо всём с юмором. Отвечай кратко (2-3 предложения) на русском.", 
              "green")
    ]

    console.print()
    console.print(Panel(
        f"[bold white]{topic}[/bold white]", 
        title="[bold blue]Тема дискуссии[/bold blue]", 
        border_style="blue"
    ))
    console.print()

    history = [{"role": "user", "content": f"Давайте обсудим: {topic}. Выскажите краткое мнение."}]

    for i in range(rounds):
        console.rule(f"[bold white]Раунд {i + 1}[/bold white]")
        console.print()
        
        for agent in agents:
            response = await agent.reply(history)
            
            panel = Panel(
                Markdown(response),
                title=f"[bold {agent.color}]{agent.name}[/bold {agent.color}]",
                border_style=agent.color,
                expand=False
            )
            console.print(panel)
            console.print()
            
          
            history.append({"role": "assistant", "content": f"{agent.name}: {response}"})
            
            await asyncio.sleep(0.5)

    console.print("[bold green]Дискуссия окончена![/bold green]")


await run_discussion("Будущее искусственного интеллекта", rounds=2)

╭──────────────────────────────────────────────── Тема дискуссии ─────────────────────────────────────────────────╮
│ Будущее искусственного интеллекта                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

───────────────────────────────────────────────────── Раунд 1 ─────────────────────────────────────────────────────

╭─────────────────────────────────────────────────── Оптимист ────────────────────────────────────────────────────╮
│ Ошибка: Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits.    │
│ Make sure your key is on the correct account or org, and if so, purchase more at                                │
│ https://openrouter.ai/settings/credits', 'code': 402}}                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Пессимист ───────────────────────────────────────────────────╮
│ Ошибка: Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits.    │
│ Make sure your key is on the correct account or org, and if so, purchase more at                                │
│ https://openrouter.ai/settings/credits', 'code': 402}}                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────── Учёный ─────────────────────────────────────────────────────╮
│ Ошибка: Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits.    │
│ Make sure your key is on the correct account or org, and if so, purchase more at                                │
│ https://openrouter.ai/settings/credits', 'code': 402}}                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────── Оптимист ────────────────────────────────────────────────────╮
│ Ошибка: Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits.    │
│ Make sure your key is on the correct account or org, and if so, purchase more at                                │
│ https://openrouter.ai/settings/credits', 'code': 402}}                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Пессимист ───────────────────────────────────────────────────╮
│ Ошибка: Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits.    │
│ Make sure your key is on the correct account or org, and if so, purchase more at                                │
│ https://openrouter.ai/settings/credits', 'code': 402}}                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────── Учёный ─────────────────────────────────────────────────────╮
│ Ошибка: Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits.    │
│ Make sure your key is on the correct account or org, and if so, purchase more at                                │
│ https://openrouter.ai/settings/credits', 'code': 402}}                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── Комик ─────────────────────────────────────────────────────╮
│ Ошибка: Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits.    │
│ Make sure your key is on the correct account or org, and if so, purchase more at                                │
│ https://openrouter.ai/settings/credits', 'code': 402}}                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Дискуссия окончена!